# LLMs and RAG — The Cutting Edge

Large Language Models are transformers scaled to an extreme: billions of parameters
trained on trillions of tokens. One model can write essays, code, translate, summarize,
and reason — no task-specific training needed.

This notebook covers how to use LLMs effectively, their limitations, and RAG — the
technique that connects LLMs to your private data. This is what's getting people hired.

---
## What Are LLMs?

LLMs are decoder-only transformers with massive parameter counts, trained on internet-scale text.

| Model | Creator | Parameters | Key Capability |
|---|---|---|---|
| GPT-4 | OpenAI | ~1.8T (rumored) | Best general reasoning |
| Claude 3.5 | Anthropic | Undisclosed | Long context, safety-focused |
| Llama 3 | Meta | 8B / 70B | Best open-source |
| Gemini 1.5 | Google | Undisclosed | 1M+ token context |
| Mistral | Mistral AI | 7B / 8x7B | Efficient, fast |

**Why they work:** at sufficient scale, predicting the next token requires understanding
grammar, facts, reasoning, coding patterns, and more. The model learns a compressed
representation of human knowledge.

**The training recipe:**
1. **Pre-training:** predict next token on trillions of words (raw internet text)
2. **Instruction tuning:** fine-tune to follow instructions ("Summarize this article")
3. **RLHF/DPO:** align with human preferences (be helpful, harmless, honest)

---
## Prompt Engineering — The Art of Asking

With LLMs, **how you ask** matters as much as **what you ask**. The prompt IS the program.

Four key techniques, from simplest to most powerful:

In [ ]:
prompts = {}

prompts['zero_shot'] = """Classify the following review as positive or negative.

Review: "The battery life is amazing but the screen quality is disappointing."

Classification:"""

prompts['few_shot'] = """Classify the following reviews as positive or negative.

Review: "Absolutely love this phone, best I've ever owned!"
Classification: positive

Review: "Terrible product, broke after one week."
Classification: negative

Review: "Decent camera but too expensive for what you get."
Classification: negative

Review: "The battery life is amazing but the screen quality is disappointing."
Classification:"""

prompts['chain_of_thought'] = """Classify the following review as positive or negative.
Think step by step about the sentiment.

Review: "The battery life is amazing but the screen quality is disappointing."

Let's think step by step:
1. "battery life is amazing" - this is positive
2. "but" - signals a contrast
3. "screen quality is disappointing" - this is negative
4. The positive and negative roughly balance, but "disappointing" suggests overall dissatisfaction

Classification: negative"""

prompts['system_prompt'] = """[System]: You are a product review analyst. Classify reviews into exactly one
of these categories: positive, negative, mixed. For mixed reviews, explain the trade-off.
Be concise — one sentence maximum.

[User]: "The battery life is amazing but the screen quality is disappointing."

[Assistant]:"""

print("=== PROMPT ENGINEERING TECHNIQUES ===\n")
for name, prompt in prompts.items():
    print(f"--- {name.upper().replace('_', ' ')} ---")
    print(prompt)
    print()

---
## Calling LLMs from Code — The API Pattern

Most LLM providers follow the same **chat completions** pattern:

```python
response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain transformers in one paragraph."},
    ],
    temperature=0.7,
    max_tokens=200,
)
answer = response.choices[0].message.content
```

Key parameters:
- `model` — which LLM to use
- `messages` — conversation history (system, user, assistant roles)
- `temperature` — creativity (0 = deterministic, 1 = creative)
- `max_tokens` — output length limit

In [ ]:
# !pip install openai

# from openai import OpenAI
# client = OpenAI(api_key="your-api-key-here")

def mock_llm_call(messages, temperature=0.7):
    """Simulates an LLM API call for demonstration."""
    user_msg = messages[-1]['content']
    
    if 'summarize' in user_msg.lower():
        return "[LLM would generate a concise summary of the provided text]"
    elif 'classify' in user_msg.lower():
        return "[LLM would classify the text based on the instructions]"
    else:
        return f"[LLM would generate a helpful response to: '{user_msg[:50]}...']"

messages = [
    {"role": "system", "content": "You are a helpful assistant that explains things simply."},
    {"role": "user", "content": "Summarize what transformers are in 2 sentences."},
]

response = mock_llm_call(messages)
print(f"System: {messages[0]['content']}")
print(f"User:   {messages[1]['content']}")
print(f"LLM:    {response}")
print("\nReplace mock_llm_call with the OpenAI client to get real responses.")

---
## Limitations of LLMs

LLMs are powerful but fundamentally flawed in specific ways:

| Limitation | Explanation | Example |
|---|---|---|
| **Hallucination** | Confidently generates false information | "The Eiffel Tower was built in 1920" (actually 1889) |
| **Knowledge cutoff** | Only knows what it was trained on | Can't answer about events after training date |
| **No private data** | Doesn't know your company's docs | "What's our Q3 revenue?" → makes something up |
| **Context window** | Limited input size | Can't process a 1000-page manual (most models) |
| **No real reasoning** | Pattern matching, not thinking | Struggles with novel logic puzzles |

**Hallucination is the critical problem.** The model doesn't know what it doesn't know —
it will generate plausible-sounding but completely fabricated answers.

The solution? **Give the LLM the right information before it answers.** That's RAG.

---
## RAG — Retrieval-Augmented Generation

RAG connects LLMs to your data. Instead of hoping the model memorized the answer,
you **retrieve** relevant documents and **feed them** to the LLM.

```
┌─────────────────────────────────────────────────────┐
│                    RAG PIPELINE                      │
│                                                      │
│  INDEXING (done once):                               │
│  Your Documents → Chunk → Embed → Store in Vector DB │
│                                                      │
│  QUERY (every question):                             │
│  User Question → Embed → Search Vector DB            │
│       → Get Top-K Relevant Chunks                    │
│       → Feed [Chunks + Question] to LLM              │
│       → Get Accurate, Grounded Answer                │
└─────────────────────────────────────────────────────┘
```

**Why it works:** the LLM doesn't need to memorize facts — it just needs to read
the right context and synthesize an answer. This eliminates hallucination for
factual questions about your data.

---
## Vector Databases and Embeddings for Search

Traditional search: keyword matching ("find documents containing 'transformer'")
Semantic search: meaning matching ("find documents about attention mechanisms")

**How it works:**
1. Encode every document into a vector using an embedding model
2. Encode the query into a vector
3. Find the nearest vectors using cosine similarity

**Vector databases** are optimized for this — they index millions of vectors
for fast nearest-neighbor search.

| Vector DB | Type | Best For |
|---|---|---|
| **FAISS** | Library (Meta) | Research, in-memory, fast |
| **ChromaDB** | Lightweight DB | Prototyping, local development |
| **Pinecone** | Cloud service | Production, managed, scalable |
| **Weaviate** | Open-source DB | Self-hosted production |
| **pgvector** | Postgres extension | Already using Postgres |

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)

def simple_embed(text):
    np.random.seed(hash(text) % 2**32)
    return np.random.randn(64)

documents = [
    "Transformers use self-attention to process sequences in parallel.",
    "BERT is a bidirectional encoder trained with masked language modeling.",
    "GPT generates text autoregressively, predicting one token at a time.",
    "Word2Vec learns static word embeddings from local context windows.",
    "TF-IDF weights words by their frequency in a document vs the corpus.",
    "RAG combines retrieval with generation for accurate answers.",
    "Fine-tuning adapts a pre-trained model to a specific downstream task.",
    "Tokenizers split text into subword units that the model can process.",
]

doc_embeddings = np.array([simple_embed(doc) for doc in documents])

print(f"Document store: {len(documents)} documents")
print(f"Embedding matrix: {doc_embeddings.shape}  ({len(documents)} docs × 64 dims)")

In [ ]:
def search(query, doc_embeddings, documents, top_k=3):
    query_vec = simple_embed(query).reshape(1, -1)
    similarities = cosine_similarity(query_vec, doc_embeddings)[0]
    top_indices = similarities.argsort()[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append((documents[idx], similarities[idx]))
    return results

query = "How does attention work in neural networks?"
results = search(query, doc_embeddings, documents)

print(f"Query: {query}\n")
print("Top 3 results:")
for doc, score in results:
    print(f"  [{score:.3f}] {doc}")

print("\nNote: with a real embedding model (not random), these would be semantically ranked.")

---
## Build a Simple RAG Pipeline

Let's build a complete RAG system from scratch. Even without a real LLM API key,
we can show the full architecture.

In [ ]:
class SimpleRAG:
    def __init__(self, chunk_size=200, overlap=50):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.chunks = []
        self.embeddings = None
    
    def add_documents(self, documents):
        for doc in documents:
            words = doc.split()
            for i in range(0, len(words), self.chunk_size - self.overlap):
                chunk = ' '.join(words[i:i + self.chunk_size])
                if chunk.strip():
                    self.chunks.append(chunk)
        
        self.embeddings = np.array([simple_embed(c) for c in self.chunks])
        print(f"Indexed {len(self.chunks)} chunks from {len(documents)} documents")
    
    def retrieve(self, query, top_k=3):
        query_vec = simple_embed(query).reshape(1, -1)
        sims = cosine_similarity(query_vec, self.embeddings)[0]
        top_idx = sims.argsort()[::-1][:top_k]
        return [(self.chunks[i], sims[i]) for i in top_idx]
    
    def query(self, question, top_k=3):
        retrieved = self.retrieve(question, top_k)
        
        context = "\n\n".join([chunk for chunk, _ in retrieved])
        
        prompt = f"""Answer the question based ONLY on the following context.
If the context doesn't contain the answer, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:"""
        return prompt, retrieved

rag = SimpleRAG(chunk_size=30, overlap=5)

knowledge_base = [
    """Transformers are neural network architectures that use self-attention mechanisms.
    They were introduced in 2017 by Vaswani et al. in the paper 'Attention Is All You Need'.
    Unlike RNNs, transformers can process all tokens in parallel, making them much faster
    to train on modern GPU hardware.""",
    
    """BERT (Bidirectional Encoder Representations from Transformers) was released by Google
    in 2018. It uses the encoder part of the transformer and is pre-trained with masked
    language modeling (MLM) and next sentence prediction (NSP). BERT is excellent for
    text classification, named entity recognition, and question answering.""",
    
    """GPT (Generative Pre-trained Transformer) was developed by OpenAI. It uses the decoder
    part of the transformer and is trained to predict the next token. GPT-3 has 175 billion
    parameters and demonstrated few-shot learning capabilities. GPT-4 further improved
    reasoning and multimodal capabilities.""",
    
    """RAG (Retrieval-Augmented Generation) was proposed by Lewis et al. in 2020. It combines
    a retriever (which searches for relevant documents) with a generator (an LLM that produces
    answers). RAG reduces hallucination by grounding LLM responses in actual source documents.
    It is widely used for question answering over private knowledge bases.""",
]

rag.add_documents(knowledge_base)

In [ ]:
questions = [
    "When were transformers introduced?",
    "What is BERT used for?",
    "How does RAG reduce hallucination?",
]

for q in questions:
    prompt, retrieved = rag.query(q)
    print(f"Q: {q}")
    print(f"Retrieved {len(retrieved)} chunks (top similarity: {retrieved[0][1]:.3f})")
    print(f"Top chunk: {retrieved[0][0][:80]}...")
    print(f"\n→ This prompt would be sent to the LLM for final answer generation.\n")
    print("-" * 70)

In [ ]:
print("=== FULL RAG PROMPT (what the LLM actually sees) ===\n")
prompt, _ = rag.query("When were transformers introduced and by whom?")
print(prompt)

---
## Embeddings for Real Semantic Search

The simple random embeddings above are just for demonstration.
In practice, you'd use a sentence transformer for real semantic similarity.

In [ ]:
# !pip install sentence-transformers

# Real embedding approach (uncomment if you have sentence-transformers installed):
#
# from sentence_transformers import SentenceTransformer
# embed_model = SentenceTransformer('all-MiniLM-L6-v2')
#
# docs = [
#     "Transformers process sequences in parallel using self-attention.",
#     "BERT understands text bidirectionally.",
#     "GPT generates text one token at a time.",
#     "The weather in Paris is lovely in spring.",
# ]
#
# embeddings = embed_model.encode(docs)
# query_vec = embed_model.encode(["How do attention mechanisms work?"])
# sims = cosine_similarity(query_vec, embeddings)[0]
#
# for doc, sim in sorted(zip(docs, sims), key=lambda x: -x[1]):
#     print(f"  [{sim:.3f}] {doc}")

print("With sentence-transformers, semantic search finds relevant documents")
print("even when they use different words to describe the same concept.")
print("\n'all-MiniLM-L6-v2' is a popular choice: 22M params, 384 dims, very fast.")

---
## LangChain Overview

**LangChain** is a framework for building LLM-powered applications. It provides
abstractions for common patterns:

| Component | What It Does |
|---|---|
| **Chains** | Sequence of steps (retrieve → format → LLM → parse) |
| **Agents** | LLMs that decide which tools to use |
| **Tools** | Functions the LLM can call (search, calculator, API) |
| **Memory** | Conversation history management |
| **Retrievers** | Vector DB search wrappers |

```python
# !pip install langchain langchain-openai chromadb

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA

# 1. Split documents into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(your_documents)

# 2. Embed and store in vector DB
vectorstore = Chroma.from_documents(chunks, OpenAIEmbeddings())

# 3. Create retrieval chain
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model="gpt-4"),
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
)

# 4. Ask questions!
answer = qa_chain.invoke("What does the contract say about termination?")
```

LangChain handles chunking, embedding, retrieval, prompt formatting, and LLM calls —
you focus on your data and use case.

In [ ]:
print("=== LangChain RAG Pattern (conceptual) ===")
print()
print("Step 1: INGEST")
print("  Load PDFs, web pages, databases, etc.")
print("  → Split into 500-token chunks with 50-token overlap")
print("  → Embed each chunk with sentence-transformers or OpenAI")
print("  → Store vectors in ChromaDB / Pinecone / FAISS")
print()
print("Step 2: RETRIEVE")
print("  User asks: 'What are the payment terms?'")
print("  → Embed the question")
print("  → Search vector DB for top-3 similar chunks")
print("  → Return: chunk_17 (sim=0.89), chunk_42 (sim=0.85), chunk_5 (sim=0.81)")
print()
print("Step 3: GENERATE")
print("  Prompt = system_prompt + retrieved_chunks + user_question")
print("  → Send to LLM (GPT-4, Claude, etc.)")
print("  → LLM generates answer grounded in YOUR documents")
print("  → No hallucination — facts come from source data")

---
## Fine-Tuning vs RAG — When to Use Which

| Dimension | RAG | Fine-Tuning |
|---|---|---|
| **Use case** | Access to specific data/documents | Change model behavior/style |
| **Data needed** | Any documents (unstructured) | Labeled examples (hundreds+) |
| **Knowledge updates** | Just update the vector DB | Retrain the model |
| **Hallucination** | Reduced (grounded in sources) | Still possible |
| **Cost** | Retrieval compute + LLM tokens | GPU training time |
| **Example** | "QA over company docs" | "Write in our brand voice" |

**Decision rule:**
- Need the model to **know specific facts**? → RAG
- Need the model to **behave differently**? → Fine-tuning
- Need both? → Fine-tune THEN add RAG

In [ ]:
decision_tree = """
                        What do you need?
                       /                 \\
                      /                   \\
           Access private data?      Change model behavior?
                  |                         |
                  v                         v
              Use RAG                  Fine-tune
         /        |        \\             |       \\
        v         v         v            v        v
    ChromaDB   Pinecone   FAISS     LoRA/QLoRA   Full FT
  (prototype) (production) (research) (efficient) (expensive)
"""

print(decision_tree)
print("Most real-world applications use RAG. It's simpler, cheaper, and more maintainable.")
print("Fine-tuning is for when you need to change HOW the model responds, not WHAT it knows.")

---
## Advanced RAG Techniques

In [ ]:
techniques = {
    "Naive RAG": {
        "how": "Chunk → embed → retrieve → generate",
        "pro": "Simple to implement",
        "con": "Chunks might miss context",
    },
    "Sentence Window Retrieval": {
        "how": "Embed individual sentences, but retrieve surrounding window",
        "pro": "Better context around the match",
        "con": "More tokens sent to LLM",
    },
    "Parent Document Retrieval": {
        "how": "Index small chunks, but return parent (larger) document",
        "pro": "Fine-grained search + full context",
        "con": "Parent docs may be large",
    },
    "Hybrid Search": {
        "how": "Combine vector similarity + keyword matching (BM25)",
        "pro": "Catches both semantic + exact matches",
        "con": "More complex pipeline",
    },
    "Re-ranking": {
        "how": "Retrieve top-20, then re-rank with a cross-encoder",
        "pro": "Much more accurate ranking",
        "con": "Slower (cross-encoder runs per pair)",
    },
}

for name, info in techniques.items():
    print(f"\n{name}:")
    for key, val in info.items():
        print(f"  {key:>4}: {val}")

---

## Summary

**The modern NLP stack:**

```
Text Preprocessing (BoW, TF-IDF)    ← Simple tasks, baselines
       ↓
Word Embeddings (Word2Vec, GloVe)    ← Semantic similarity
       ↓
Transformers (BERT, GPT)             ← State-of-the-art understanding/generation
       ↓
LLMs (GPT-4, Claude, Llama)          ← General intelligence
       ↓
RAG (LLM + your data)                ← Production applications
```

**Key takeaways:**
- LLMs are massive transformers trained on internet text — one model, many tasks
- Prompt engineering is the new programming — how you ask determines what you get
- RAG = retrieval + generation — grounds LLMs in your data, reduces hallucination
- Vector databases enable semantic search at scale
- RAG for knowledge, fine-tuning for behavior
- LangChain/LlamaIndex simplify building RAG applications

**This is the most in-demand skillset in AI right now.** Companies need people who can
build RAG pipelines over their data — it's the bridge between powerful LLMs and real business value.